# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 37.0 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 9.5 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sample_biology.xlsx')

In [ ]:
data.head()

,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot
0,9998,بنك أسئلة الفصل الدراسي الأول-2.pdf - Google D...,Jordan,STEM,Biology,High,إذا كان عدد جزيئات مستقبل الالكترونات النهائي ...,NaN,D,24,18,12,36,NaN,0
1,10828,https://docs.google.com/forms/d/e/1FAIpQLSfOH9...,Egypt,STEM,Biology,High,تتحول الخلايا ..... إلى خلايا بلعمية فى بعض ال...,NaN,B,الخلايا التائية,الخلايا وحيدة النواة,الخلايا الحامضية,الخلايا القاتلة الطبيعية,NaN,0
2,11086,exams-28a8e986-7722-11ea-9116-54bef70b159e,NaN,STEM,Biology,High,ما اسم الوعاء الدموي الذي يحمل الدم المؤكسج بع...,NaN,B,الصمام,الشرايين,الشعيرات,الأوردة,NaN,0
3,9740,https://drive.google.com/file/d/1XyNWikrWZ01Gi...,Palestine,STEM,Biology,High,خلية تحتوي على 16 كروموسوم في المرحلة النمو ال...,NaN,B,8,16,32,35,NaN,0
4,10862,https://docs.google.com/forms/d/e/1FAIpQLScuuw...,Egypt,STEM,Biology,High,كل المواد التالية وسائل مناعة تركيبية موجودة س...,NaN,D,شموع,شعيرات,اشواك,اصماغ,NaN,0


# Zero Shot

In [ ]:
content = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer
'''
zero_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''The question you have to answer:
    Question:
    {text}
    Options:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    Answer:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
b) Answer,17
c) Answer,14
d) Answer,11
d),5
a) Answer\n,3
...,...
b) الدرقيه,1
c) عمودية ع محور الليفة العضلية,1
b) انزيمات,1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "a)" in pr:
    nor_pre.append("A")
  elif "b)" in pr:
    nor_pre.append("B")
  elif "c)" in pr:
    nor_pre.append("C")
  elif "d)" in pr:
    nor_pre.append("D")
  else:
    nor_pre.append("Unclassified")

pred_zero['Normalized Category'] = nor_pre

In [ ]:
pred_zero['question'] = data['Question']
pred_zero.to_excel('Qwen-QA-Biology-ZeroShot.xlsx', index = False)

In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_zero['Normalized Category'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3333    0.4722    0.3908        36
           B     0.4375    0.4746    0.4553        59
           C     0.5000    0.4032    0.4464        62
           D     0.4000    0.3256    0.3590        43

    accuracy                         0.4200       200
   macro avg     0.4177    0.4189    0.4129       200
weighted avg     0.4301    0.4200    0.4202       200



# Pred Few Shot

In [ ]:
content = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer

Example 1
Question:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

Options:
a) 220 أوم
b) 202 أوم
c) 100 أوم
d) 484 أوم

Answer:
d) 484 أوم

Example 2
Question:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

Options:
a) جلوكوزيدات
b) مستقبلات
c) أحماض أمينية غير بروتينية
d) سيفالوسبورين

Answer:
b) مستقبلات

Example 3
Question:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

Options:
a) ناتج الطرح
b) المطروح
c) المطروح منه
d) لاشيء مما سبق

Answer:
a) ناتج الطرح
'''

few_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''The question you have to answer:
    Question:
    {text}
    Options:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    Answer:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
d),10
c),6
b),5
d) جميع الإجابات السابقة صحيحة,2
d) أول إجابتين,2
...,...
a) العباره الاولى صحيحه والثانيه خطا,1
b) 6,1
"b) Number of stomata is few, cuticle layer is thick, and root hairs are abundant",1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "a)" in pr:
    nor_pre.append("A")
  elif "b)" in pr:
    nor_pre.append("B")
  elif "c)" in pr:
    nor_pre.append("C")
  elif "d)" in pr:
    nor_pre.append("D")
  else:
    nor_pre.append("Unclassified")

pred_few['Normalized Category'] = nor_pre

In [ ]:
pred_few['question'] = data['Question']
pred_few.to_excel('Qwen-QA-Biology-FewShot.xlsx', index = False)

In [ ]:
pred_few['Normalized Category'].value_counts()

,count
Normalized Category,
B,63
C,51
A,50
D,36


In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_few['Normalized Category'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3400    0.4722    0.3953        36
           B     0.4286    0.4576    0.4426        59
           C     0.5098    0.4194    0.4602        62
           D     0.4167    0.3488    0.3797        43

    accuracy                         0.4250       200
   macro avg     0.4238    0.4245    0.4195       200
weighted avg     0.4353    0.4250    0.4260       200



# CoT

In [ ]:
content = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer

Follow these reasoning steps to determine the correct answer:

Step 1: Understand the question
Carefully read and comprehend the question. Identify what is being asked, including the underlying need and the context.

Step 2: Analyze the options
Carefully read each of the four options. Consider the meaning of each option and how it relates to the question.

Step 3: Evaluate the options
Apply your knowledge and reasoning to assess each option. Eliminate clearly incorrect options. Compare the remaining options to determine which is most accurate.

Step 4: Select the best answer
Based on your evaluation in the previous steps, choose the single most logical and correct option.

Step 5: Output the answer
Write the correct answer in this exact format:
a/b/c/d) Answer

Example 1
Question:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

Options:
a) 220 أوم
b) 202 أوم
c) 100 أوم
d) 484 أوم

Thinking:
- Step 1: Understand the question
We are asked to calculate the electrical resistance of a lamp rated at 220 volts and 100 watts.
- Step 2: Analyze the options
The resistance must be computed based on the given voltage and power. The options are 220 ohms, 202 ohms, 100 ohms, and 484 ohms.
- Step 3: Evaluate the options
We know the formula for electrical resistance in terms of voltage and power is:
R = V^2/P
Substitute the given values:
R = 220^2/100 = 48400/100 = 484 ohms
- Step 4: Select the correct answer
From the options, 484 أوم corresponds to option d.

Final Answer:
d) 484 أوم

Example 2
Question:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

Options:
a) جلوكوزيدات
b) مستقبلات
c) أحماض أمينية غير بروتينية
d) سيفالوسبورين

Thinking:
- Step 1: Understand the question
We are asked: Which of these functions similarly to white blood cells in humans?
White blood cells (WBCs) are part of the immune system — they fight infections by detecting and neutralizing pathogens (bacteria, viruses, etc.).
- Step 2: Analyze the options
a) Glycosides (جلوكوزيدات): These are sugar-based compounds, not immune-related.
b) Receptors (مستقبلات): Receptors help in detecting and recognizing substances (including pathogens), playing a role in immunity and signaling.
c) Non-protein amino acids (أحماض أمينية غير بروتينية): These are rare metabolic intermediates, not directly involved in immune function.
d) Cephalosporins (سيفالوسبورين): These are antibiotics; they kill bacteria but are not a cellular component of the immune system.
- Step 3: Evaluate the options
Which of these functions similarly to WBCs — that is, by detecting or responding to pathogens?
Receptors help the immune system recognize invaders — similar to how WBCs detect and respond to pathogens.
- Step 4: Select the correct answer
The option that most closely mimics the immune detection function of WBCs is receptors (مستقبلات).

Final Answer:
b) مستقبلات

Example 3
Question:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

Options:
a) ناتج الطرح
b) المطروح
c) المطروح منه
d) لاشيء مما سبق

Thinking:
- Step 1: Understand the question
We are asked: What is the name of the number 50 in the subtraction operation 70 – 20 = 50?
In subtraction:
The first number (70) is called the minuend (المطروح منه).
The second number (20) is called the subtrahend (المطروح).
The result (50) is called the difference (ناتج الطرح).
- Step 2: Analyze the options
a) ناتج الطرح: "Result of subtraction" — this is the correct mathematical term for the outcome of the subtraction (difference).
b) المطروح: "The number being subtracted" — this is 20, not 50.
c) المطروح منه: "The number from which another number is subtracted" — this is 70, not 50.
d) لاشيء مما سبق: "None of the above" — not applicable since option (a) is correct.
- Step 3: Evaluate the options
The number 50 is the result of the subtraction, also known as the difference.
- Step 4: Select the correct answer
The correct answer is ناتج الطرح.

Final Answer:
a) ناتج الطرح
'''

cot_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''The question you have to answer:
    Question:
    {text}
    Options:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
d),5
a),5
c) 14,2
c),2
c) عمودية ع محور الليفة العضلية,2
...,...
a) العباره الاولى صحيحه والثانيه خطا,1
c) 3,1
"b) Answer\n\nReasoning:\n- For a plant to be more physiologically robust, it needs to be able to efficiently absorb water and nutrients, and have a strong structure to withstand environmental stresses.\n- Option (b) describes a plant with a few stomata (thgors), a thick cuticle (kiyatin layer), and many root hairs. This combination suggests the plant can minimize water loss (due to fewer stomata), retain moisture (due to a thick cuticle), and efficiently absorb water and nutrients (due to many root hairs).\n- The other options either describe characteristics that would make the plant less efficient (like having many stomata leading to higher water loss or a thin cuticle leading to easier water loss) or do not provide the necessary combination for robustness.",1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "a)" in pr:
    nor_pre.append("A")
  elif "b)" in pr:
    nor_pre.append("B")
  elif "c)" in pr:
    nor_pre.append("C")
  elif "d)" in pr:
    nor_pre.append("D")
  else:
    nor_pre.append("Unclassified")

pred_cot['Normalized Category'] = nor_pre

In [ ]:
pred_cot['question'] = data['Question']
pred_cot.to_excel('Qwen-QA-Biology-CoT.xlsx', index = False)

In [ ]:
pred_cot['Normalized Category'].value_counts()

,count
Normalized Category,
B,63
C,51
A,50
D,36


In [ ]:
print(classification_report(y_true, pred_cot['Normalized Category'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3400    0.4722    0.3953        36
           B     0.4286    0.4576    0.4426        59
           C     0.5098    0.4194    0.4602        62
           D     0.4167    0.3488    0.3797        43

    accuracy                         0.4250       200
   macro avg     0.4238    0.4245    0.4195       200
weighted avg     0.4353    0.4250    0.4260       200

